# Module 25 — Textual

The last of the five presentations, and the same numbers again — this time in the
terminal you are already sitting in.

```console
uv run python 25_textual/app.py
```

`q` quits, and so does Ctrl+C. Module 24's window could not manage that; here the
terminal is still the terminal.

`app.py` imports `24_tkinter/logic.py`. Not a copy of it — **the same file.** Module
24's argument for keeping the formatting out of the callbacks was that a callback
cannot be tested. This module is the second payment on that argument, and it arrived
without any work.

Nothing below needs a terminal. `App.run_test()` runs the app with the screen in
memory, and `Pilot` is the object that pretends to be a user.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))

from app import SensorApp  # noqa: E402
from textual.widgets import Button, DataTable, Static  # noqa: E402

found = []

app = SensorApp()
async with app.run_test() as pilot:
    await pilot.pause()
    found.append(("location", app.location))
    found.append(("rows", app.query_one(DataTable).row_count))
    found.append(("status", str(app.query_one("#status", Static).render())))
    found.append(("first row", app.query_one(DataTable).get_row_at(0)))

for name, value in found:
    print(f"  {name:10} {value}")

**Why the results are collected in a list and printed afterwards.** Measured: a
`print` *inside* the `async with` block produces nothing in a notebook. Textual has
taken over the output stream — it is drawing a screen there — and hands it back when
the app stops.

Which is the module's thesis arriving three lines in, by accident: **Textual owns the
screen.** In a plain script the prints do come out, because there is no notebook
machinery in between; here there is, and the app wins. Every cell below collects and
then prints.

**About `async` and `await`.** `run_test()` is an asynchronous context manager, so
`async with` and `await` are how you use it.

In a *script* that means an `async def` started with `asyncio.run(...)`, which is what
`exercises/exercise_01.py` does. In a *notebook* it does not, and the difference is
worth knowing: the kernel is already running an event loop, so `await` and `async
with` work at the top level of a cell — as they do above — and `asyncio.run(...)` in a
cell fails with `RuntimeError: asyncio.run() cannot be called from a running event
loop`. Measured. Same code, two hosts, two spellings.

This course does not teach asyncio and you do not need it here. Read `await x` as "call x, and let the event loop have a turn
while it works" — the same event loop idea as module 24, except that this one is
asyncio's rather than Tk's C code. `await pilot.pause()` means "let everything pending
happen before I look".

## 1. The layout is data

Here is `compose` from `app.py`:

```python
def compose(self) -> ComposeResult:
    yield Static("Sensor readings", id="title")
    with Horizontal(id="controls"):
        yield Label("location")
        yield Button(self.location, id="location")
        yield Label("limit")
        yield Input(value=f"{LIMIT:.1f}", id="limit")
    yield DataTable(id="table")
    yield Static("", id="status")
```

Not one word about where anything goes, how wide it is or what colour. That is in
`app.tcss` next to it — a real stylesheet, because Textual parses CSS:

```css
#table {
    height: 1fr;
    border: round $primary;
}
```

Module 24 arranged its widgets with `.pack()` calls inside `__init__`: code, in a
method, in a class. Change a width there and you edit Python. Change it here and you
edit a file with no behaviour in it at all.

And look at what `compose` is.

In [ ]:
import inspect

# It has `yield` in it. What does that make it?
assert inspect.isgeneratorfunction(SensorApp.compose) == ...

In [ ]:
print("isgeneratorfunction:", inspect.isgeneratorfunction(SensorApp.compose))
# Module 13: calling a generator function runs none of its body.
print("calling it gives   :", type(SensorApp().compose()).__name__)

**Module 13, in a place you would not have looked for it.** `compose` does not build
the interface — it *describes* it, and Textual is the consumer of what it yields.
Calling it on its own produces no widgets at all, which is why doing so outside a
running app is harmless.

That is also what makes the CSS possible. A function that yielded widgets *and* placed
them would have the layout inside it; one that only yields leaves the placing to
whoever consumes the sequence.

## 2. A handler is found by name

```python
def on_button_pressed(self, event: Button.Pressed) -> None: ...
```

Nothing registers that. No `command=`, no decorator: the method is called because of
what it is *called*. Keys work the same way — `BINDINGS = [("n", "next_location",
"Next location")]` finds `action_next_location`.

Which is the third *mechanism* the course has shown — a decorator (21 and 23), an
argument passed at construction (24), and the method's name (25) — and it is
convenient until it is not.

In [ ]:
from textual.app import App, ComposeResult


class Typo(App[None]):
    def compose(self) -> ComposeResult:
        yield Static("start", id="row")
        yield Button("go", id="go")

    # `on_button_press`. Correct signature, correct body, right idea.
    def on_button_press(self, event: Button.Pressed) -> None:
        self.query_one("#row", Static).update("handler ran")


result = []
typo = Typo()
async with typo.run_test() as pilot:
    await pilot.pause()
    result.append(str(typo.query_one("#row", Static).render()))
    await pilot.click("#go")
    await pilot.pause()
    result.append(str(typo.query_one("#row", Static).render()))

print("before and after the click:", result)
print("the name Textual looks for:", Button.Pressed.handler_name)

No exception, no warning, and a button that does nothing — because a missing handler
is the **normal** case. Most messages have no handler on most widgets, so Textual
cannot treat an absent one as an error. The lookup is `getattr(self,
message.handler_name, None)` and it found nothing, exactly as it does a thousand times
a second for messages nobody cares about.

`Button.Pressed.handler_name` is the answer rather than a recollection of it. Every
message class carries the method name it will be delivered to, and that attribute is
the first thing to check when a handler is silently not firing.

### Four ways to say "call this", and three silent failures

Four module answers, three mechanisms between them. Three of the four can fail
without saying anything, and each fails somewhere different.

In [ ]:
import tkinter

sys.path.append(str(Path.cwd().parent / "24_tkinter"))
from display import use_bundled_tcl  # noqa: E402

use_bundled_tcl()

from flask import Flask  # noqa: E402

# 24 -- a mistake in the value passed. `note()` is valid and returns None, and Tk
# accepts None as "this widget has no command".
root = tkinter.Tk()
root.withdraw()
pressed = []
wrong = tkinter.Button(root, text="x", command=(lambda: pressed.append(1))())
wrong.pack()
root.update()
wrong.invoke()
print("24 tkinter: presses recorded:", pressed, "| cget('command'):", repr(wrong.cget("command")))
root.destroy()

# 21 -- a mistake in a string inside it. The decorator worked perfectly.
web = Flask(__name__)


@web.get("/summry")
def summary() -> str:
    return "ok"


client = web.test_client()
print("21 flask  : /summary ->", client.get("/summary").status_code, end="")
print(" | /summry ->", client.get("/summry").status_code, end="")
print(" | url_map:", [str(rule) for rule in web.url_map.iter_rules() if "sum" in str(rule)])

| module | how | the silent failure | what you can ask |
| --- | --- | --- | --- |
| 21 Flask | `@app.get("/summary")` | a typo in the **path**: `/summry` answers 200 | `app.url_map` |
| 22 Streamlit | nothing to register | — | — |
| 24 tkinter | `command=self.refresh` | a typo in the **value**: `self.refresh()` passes `None` | `widget.cget("command")` |
| 25 Textual | the method's **name** | a typo in the **name**: never called | `Button.Pressed.handler_name` |

In each case the mistake is **a valid value in a position the framework has no
expectations for**. A valid expression, a valid method, a valid string. Which is why
no type checker and no linter catches any of the three: nothing is mistyped, nothing
is unused, nothing is unreachable.

**Streamlit is the exception, and it is a consequence rather than a win.** There is
nothing to register there because the script re-runs and a widget call *returns* its
current value. No name is looked up and no function is stored — so the bug class is
absent because the mechanism is absent, and what it cost was the ability to say "run
this when that happens" at all. That is why `st.session_state` had to exist.

## 3. A reactive attribute redraws by itself

```python
limit: reactive[float] = reactive(LIMIT, init=False)
```

`app.limit = 90.0` is an ordinary assignment and `app.limit` is a **float** — no
wrapper, unlike module 24's `StringVar`, which held a string and needed `.get()`.
What is not ordinary is that Textual then calls `watch_limit`, so the redraw is a
*consequence* of the assignment rather than something the assigning code has to
remember.

In [ ]:
seen = []
watched = SensorApp()
async with watched.run_test() as pilot:
    await pilot.pause()

    original = watched.watch_limit

    def counted() -> None:
        seen.append(watched.limit)
        original()

    watched.watch_limit = counted  # type: ignore[method-assign]

    watched.limit = 90.0
    await pilot.pause()
    a = str(watched.query_one("#status", Static).render())

    watched.limit = 20.0
    await pilot.pause()
    b = str(watched.query_one("#status", Static).render())

print("type of app.limit :", type(watched.limit).__name__)
print("status at 90      :", a)
print("status at 20      :", b)
print("watcher fired for :", seen)

Two assignments, two watcher calls — and that is what `init=False` in `app.py` buys.

Without it Textual calls the watcher **once at construction time**: measured, it runs
with the old and the new value both 85.0, before anything is on screen. `redraw` would
then go looking for a `DataTable` that `compose` has not yielded yet, and the app would
fail to start with `NoMatches` — which is exactly how this file's first draft behaved.

So the count without `init=False` is not three. There is no count, because there is no
app. The startup
redraw belongs in `on_mount`, the first moment the widgets exist.

Not guessable, and worth writing down.

## The test client that works, and why it can

Modules 21, 22 and 23 each had one. Module 24 had **none**, because its input is
events from the window manager: the same simulated keypress answered differently with
the window visible, off-screen and withdrawn.

Here it works, and the reason is one sentence: **Textual owns the screen.** It is a
grid of characters that Textual allocates and fills, and nothing else has an opinion
about it. You can even choose how big it is.

In [ ]:
sizes = []
small = SensorApp()
async with small.run_test(size=(40, 8)) as pilot:
    await pilot.pause()
    sizes.append(("screen", small.screen.size))
    sizes.append(("table region", small.query_one(DataTable).region))

for name, value in sizes:
    print(f"  {name:14} {value}")

Forty characters by eight, because the size is Textual's to decide. No window manager
was asked and none could have refused.

That is also the property that makes it work over an ssh connection: what goes down
the wire is characters. Nothing here needs `DISPLAY`, and none of the measurements in
this notebook used one.

### Where the simulation is not naive

`app.py` cycles the location on the `n` key and on the button, through the same
method. Drive it both ways.

In [ ]:
import asyncio


async def drive(how: str, gap: float = 0.0) -> list[str]:
    """Five actions, and the location after each."""
    driven = SensorApp()
    seen_here: list[str] = []
    async with driven.run_test() as inner:
        await inner.pause()
        for _ in range(5):
            await (inner.press("n") if how == "key" else inner.click("#location"))
            await inner.pause()
            if gap:
                await asyncio.sleep(gap)
            seen_here.append(driven.location)
    return seen_here


def moves(seq: list[str]) -> int:
    """How many of the five actually changed anything."""
    return sum(1 for was, now in zip(["Hall", *seq[:-1]], seq) if was != now)


by_key = await drive("key")
by_key_again = await drive("key")
by_click = await drive("click")
spaced = await drive("click", gap=0.3)

print("five presses of n        :", by_key)
print("the same again           :", by_key_again)
print("identical?               :", by_key == by_key_again)
print("five clicks, no delay    :", moves(by_click), "registered")
print("five clicks, 0.3 s apart :", moves(spaced), "registered")

Keys: the same list twice. Clicks with no delay: **three** of five.

That is not a flaw in Textual and not a flaw in `Pilot`. Two clicks in quick
succession on one widget are a **double-click** — one `Button.Pressed`, not two, which
is what a real user's double-click does.

Worse for a test than the count suggests: *which* pairs get folded varies between
runs, because it depends on where the wall clock falls relative to the double-click
window. Measured over five runs, the list came out three different ways while the
count was 3 every time.

**So: press keys, or space your clicks, and never assert on a sequence whose order a
clock decides.** Module 22 said a flaky test is worse than no test, and this is the
same trap with a different door.

Note the difference from module 24 all the same. There the answer changed with the
**environment** and could not be relied on at all. Here it is a documented, reproducible
property of the toolkit, and a reliable measurement is available next to it.

## The same bug, the opposite answer

Modules 24 and 25 get handed one identical mistake: a handler that raises
`ValueError`. Predict what happens to the app.

In [ ]:
class Boom(App[None]):
    def compose(self) -> ComposeResult:
        yield Static("start", id="row")
        yield Button("boom", id="boom")

    def on_button_pressed(self, event: Button.Pressed) -> None:
        raise ValueError("the handler is broken")


boom = Boom()
raised = False
running_after = None
try:
    async with boom.run_test() as pilot:
        await pilot.pause()
        await pilot.click("#boom")
        await pilot.pause()
        running_after = boom.is_running
except Exception as exc:
    raised = type(exc).__name__

print("app still running   :", running_after)
print("run_test re-raised  :", raised)

In [ ]:
# Module 24 measured a tkinter callback raising: exit code 0, traceback on stderr.
# What is the exit code of a Textual script whose handler raises and is not caught?
assert 1 == ...

Measured, both, from one identical bug:

| | tkinter (24) | Textual (25) |
| --- | --- | --- |
| the app afterwards | still running | stopped |
| the traceback | stderr, `Exception in Tkinter callback` | propagated, names your frame |
| the caller | `invoke()` returns normally | re-raised on leaving `run_test()` |
| the exit code | **0** | **1** |

**Neither is a bug.** tkinter assumes the process exists to serve **a person who is
present and has state in it**, so one broken button must not take away a window
somebody has been typing into for twenty minutes. Textual assumes the process is **an
interface it is answerable for**, and a screen that no longer reflects what the
program believes is worse than no screen.

Which assumption is right depends on how the program was started. Double-clicked from
a desktop: tkinter's, because nobody is reading stderr and no supervisor will restart
anything. Started from a Makefile: Textual's, because something is reading the exit
code and "the interface silently stopped working" has to become "this step failed".

Neither toolkit knows which case it is in. You do — and exercise 08 asks what you
would add to the tkinter program so its failures are as findable as this one's, and
what you would deliberately not copy.

## All five, side by side

| | Flask (21) | Streamlit (22) | FastAPI (23) | tkinter (24) | Textual (25) |
| --- | --- | --- | --- | --- | --- |
| the caller | a browser | a browser | a program | a person here | a person in a terminal |
| the layout | your HTML | Streamlit's | none | `.pack()` calls | a CSS file |
| a callback | a decorator | none | a decorator | `command=` | the method's name |
| a test client | `test_client()` | `AppTest` | `TestClient` | **none** | `run_test()` |
| needs a display | no | no | no | **yes** | no |
| needs a port | yes | yes | yes | no | no |
| works over ssh | with tunnelling | with tunnelling | with tunnelling | no | **yes** |

**The heuristic, finally: what can you assume about the far end?** A browser and a
port you can reach, and it is one of the first three. A screen on this machine, and it
is tkinter. Nothing but a terminal, and it is this one — which is the honest answer
more often than it looks, because a terminal is what you have on every machine you can
reach at all.

The "with tunnelling" column is the one people forget. Flask, Streamlit and FastAPI
need no display, which is true and not the difficulty: they need a **port you can
reach**, and between you and a server there is usually a firewall that allows 22 and
nothing else. `ssh -L 8501:localhost:8501` solves it, and is a second thing to set up,
explain, and redo after every reconnection.

## Two more, which this course does not cover

**PyQt6** and **PySide6** both wrap Qt, the same C++ library, and the code you write
is nearly identical. What decides between them is the licence: **PyQt6 is GPL or a
paid commercial licence; PySide6 is LGPL.**

For an internal tool nobody outside the company receives, either is fine. For anything
shipped to a customer, PyQt6 means publishing your source under the GPL or buying a
licence, and PySide6 does not — so **PySide6 is the one to evaluate first.** The three
questions that settle it are ones you can answer and your legal department cannot:
does this program leave the building, in what form, and does anyone receive a copy of
the binary?

---

`exercises/` is next: seven files to fill in and two to think through. None of them
needs a terminal.

That is Part 5 — one analysis, five presentations, and every difference between them
measured rather than asserted. Module 26 is the final project: everything at once,
built on your own.